# Preprocessing: Trem2 × 5xFAD mouse-brain MERFISH

This notebook downloads and prepares the multi-section example dataset: a MERFISH cohort of mouse brain sections from four genotypes. It writes `~/spectrast_data/trem2_5xfad.h5ad`, which contains:

- `X`: volume-normalised, log1p-transformed expression, cells × genes;
- `obsm["spatial"]`: cell centroids in µm;
- `obs`: `gen_fine` (section), `gen_coarse` (genotype), `region_labels_coarse` (brain region), `plaque_distance`, `volume` (segmented cell volume, µm³), `amyloid` (amyloid⁺ or amyloid⁻ genotype) and `section_id` (the replicate unit for comparisons between conditions).

**Requirements.** `scanpy`, `anndata`, `h5py` and `requests`. The download is a single file of about 5.5 GB; processing needs a few GB of memory.

In [ ]:
import os, time
import numpy as np, pandas as pd, scipy.sparse as sp
import anndata as ad, h5py, requests
from anndata.io import read_elem

DATA_DIR = os.path.expanduser("~/spectrast_data")
RAW = os.path.join(DATA_DIR, "trem2_raw")                 # the download saves to this dir
OUT = os.path.join(DATA_DIR, "trem2_5xfad.h5ad")          # output of this notebook
TMP = os.path.join(DATA_DIR, "trem2_5xfad.tmp.h5ad")      # written first, checked, then moved to OUT
NEW = os.path.join(DATA_DIR, "trem2_5xfad.new.h5ad")      # kept instead if it differs from an existing OUT
os.makedirs(RAW, exist_ok=True)

def short(path):
    """Print paths relative to the home directory."""
    return path.replace(os.path.expanduser("~"), "~", 1)

## 1. Source

The data are from

> Johnston KG, Berackey BT, Tran KM, Gelber A, Yu Z, MacGregor GR, Mukamel EA, Tan Z, Green KN, Xu X. Single-cell spatial transcriptomics reveals distinct patterns of dysregulation in non-neuronal and neuronal cells induced by the Trem2<sup>R47H</sup> Alzheimer's risk gene mutation. *Molecular Psychiatry* **30**, 461–477 (2025; published online 2024). https://doi.org/10.1038/s41380-024-02651-0

The study imaged 19 coronal half-sections from 15 mice aged 12 months, from four genotypes: wild type (WT), Trem2<sup>R47H</sup> (Trem2), 5xFAD, and Trem2<sup>R47H</sup>;5xFAD, on a Vizgen MERSCOPE with a 300-gene panel. The paper's data availability statement places the raw and processed data in the Brain Image Library (BIL) under identifier `b72faf9d87d7fc00`:

> https://download.brainimagelibrary.org/b7/2f/b72faf9d87d7fc00/

BIL splits the identifier into its first two byte pairs (`b7/2f/`) and then the full identifier. Alongside the processed AnnData object, the deposit holds the raw imaging data, which we do not need.

In [ ]:
# The processed cell x gene object in the Brain Image Library deposit (raw counts in layer "RNA")
BIL = "https://download.brainimagelibrary.org/b7/2f/b72faf9d87d7fc00/"
SRC_URL = BIL + "MERFISH_Data.h5ad"
SRC = os.path.join(RAW, "MERFISH_Data.h5ad")

## 2. Download

The download goes to a `.part` file that is renamed when complete, so a file is never truncated. If the connection drops, the next attempt resumes from where the last one stopped.

In [ ]:
def download(url, dst, timeout=120, tries=10, chunk=16 * 2**20):
    """Download `url` to `dst` via `<dst>.part`, resuming after a dropped connection."""
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f"{os.path.basename(dst)}: already present ({os.path.getsize(dst) / 1e9:.2f} GB)")
        return
    part = dst + ".part"
    total = int(requests.head(url, timeout=timeout, allow_redirects=True).headers["Content-Length"])
    t0 = time.perf_counter()
    for attempt in range(1, tries + 1):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        try:
            headers = {"Range": f"bytes={have}-"} if have else {}
            with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                mode = "ab" if (have and r.status_code == 206) else "wb"   # 200 = no resume: start over
                with open(part, mode) as f:
                    for block in r.iter_content(chunk):
                        f.write(block)
            if os.path.getsize(part) == total:
                break
        except (requests.RequestException, OSError) as exc:          # a timeout or a dropped connection
            got = os.path.getsize(part) if os.path.exists(part) else 0
            print(f"attempt {attempt} stopped at {got / 1e9:.2f} GB ({type(exc).__name__}); retrying")
    else:
        raise RuntimeError(f"download incomplete after {tries} attempts")
    os.replace(part, dst)
    dt = time.perf_counter() - t0
    print(f"{os.path.basename(dst)}: {total / 1e9:.2f} GB in {dt / 60:.1f} min ({total / 1e6 / dt:.0f} MB/s)")


download(SRC_URL, SRC)          # skipped when the file is already present

## 3. Load

The deposited object carries raw counts in the `RNA` layer, per-cell `volume`, genotype (`gen_coarse`) and section (`gen_fine`) labels, region annotations and plaque distances in `obs`, and cell centroids in `obsm["spatial"]`. It also holds normalised matrices, embeddings and a neighbour graph that we do not need, so lets only read in what we need for the tutorial.

In [4]:
with h5py.File(SRC, "r") as f:
    print("layers:", list(f["layers"].keys()), "| obsm:", list(f["obsm"].keys()))
    C = read_elem(f["layers/RNA"])            # raw counts
    obs = read_elem(f["obs"])
    var = read_elem(f["var"])
    xy = read_elem(f["obsm/spatial"])
print(f"{C.shape[0]:,} cells x {C.shape[1]} genes; counts are whole numbers: {bool(np.all(C[:10000] == np.round(C[:10000])))}")

layers: ['Normalized_Expression', 'RNA'] | obsm: ['X_pca', 'X_pca_harmony', 'X_umap', 'spatial']


432,794 cells x 300 genes; counts are whole numbers: True


## 4. Quality control and volume normalisation

The deposited object has already been processed and filtered by the authors: no cell has fewer than 26 counts or 5 detected genes or a volume below 50 µm³, and every one of the 300 genes is detected. We start from their raw-count layer and add a light pass on top. We drop cells with fewer than 30 counts or fewer than 5 detected genes, and cells in the extreme 1% tails of segmented volume on either side (likely segmentation errors). We then normalise for cell size by volume rather than by total counts: each cell's counts are multiplied by (median volume / cell volume), followed by `log1p`. Larger segmented cells capture more transcripts regardless of expression level, so physical volume is the imaging analogue of a library-size factor (cf. Moffitt et al., *Science* 2018). Note, you can use typical expressed based normalization here as well. 

In [5]:
C = (C.tocsr() if sp.issparse(C) else sp.csr_matrix(C)).astype(np.float32)
ng = np.asarray((C > 0).sum(1)).ravel()
vol = obs["volume"].values.astype(float)
tot = np.asarray(C.sum(1)).ravel()
vlo, vhi = np.percentile(vol, [1, 99])
keep = (tot >= 30) & (ng >= 5) & (vol >= vlo) & (vol <= vhi)
print(f"volume 1st-99th percentile: {vlo:.0f}-{vhi:.0f} µm³")
print(f"fail counts: {int((tot < 30).sum()):,}; fail genes: {int((ng < 5).sum()):,}; "
      f"outside volume range: {int(((vol < vlo) | (vol > vhi)).sum()):,}")
print(f"QC: keep {int(keep.sum()):,} of {len(keep):,} cells ({keep.mean():.1%})")
obs, C, vol, xy = obs[keep].copy(), C[keep], vol[keep], xy[keep]
Xn = C.multiply(np.median(vol) / vol[:, None]).tocsr()    # volume normalisation
Xn.data = np.log1p(Xn.data)

volume 1st-99th percentile: 60-1443 µm³
fail counts: 3,969; fail genes: 0; outside volume range: 8,656
QC: keep 420,563 of 432,794 cells (97.2%)


## 5. Condition labels

Two genotypes carry the 5xFAD amyloid transgenes, so we label sections as amyloid⁺ (5xFAD, Trem2_5xFAD) or amyloid⁻ (WT, Trem2). Each `gen_fine` value is one section, and we copy it to `section_id`, the replicate unit for comparisons between conditions.

In [6]:
adata = ad.AnnData(X=Xn, obs=obs, var=var)
adata.obsm["spatial"] = np.asarray(xy, np.float32)
AMY = {"5xFAD": "amyloid+", "Trem2_5xFAD": "amyloid+", "WT": "amyloid-", "Trem2": "amyloid-"}
adata.obs["amyloid"] = adata.obs["gen_coarse"].map(AMY)
adata.obs["section_id"] = adata.obs["gen_fine"].astype(str)
adata.obs = adata.obs[["gen_fine", "gen_coarse", "region_labels_coarse", "plaque_distance",
                       "volume", "amyloid", "section_id"]].copy()
adata.obs.groupby(["amyloid", "gen_coarse"], observed=True).agg(
    sections=("section_id", "nunique"), cells=("section_id", "size"))

sections   cells
amyloid  gen_coarse                   
amyloid+ 5xFAD               5  112403
         Trem2_5xFAD         5  110317
amyloid- Trem2               5  111879
         WT                  4   85964

## 6. Summary



In [7]:
from scipy.spatial import cKDTree
print(f"{adata.n_obs:,} cells x {adata.n_vars} genes in {adata.obs['section_id'].nunique()} sections; "
      f"X is {type(adata.X).__name__}, {adata.X.dtype}")
rows = []
for s, idx in adata.obs.groupby("section_id", observed=True).indices.items():
    p = adata.obsm["spatial"][idx].astype(float)
    nn = cKDTree(p).query(p, k=2)[0][:, 1]
    ext = np.ptp(p, axis=0)
    rows.append((s, adata.obs["gen_coarse"].iloc[idx[0]], len(idx), round(ext[0]), round(ext[1]),
                 round(float(np.median(nn)), 1)))
pd.DataFrame(rows, columns=["section_id", "genotype", "cells", "extent_1_um", "extent_2_um",
                            "median_nn_um"]).set_index("section_id")

420,563 cells x 300 genes in 19 sections; X is csr_matrix, float64


,genotype,cells,extent_1_um,extent_2_um,median_nn_um
section_id,,,,,
5xFAD1,5xFAD,23552,5407,5639,14.5
5xFAD2,5xFAD,24986,5100,6031,14.7
5xFAD3,5xFAD,22375,5239,7250,15.9
5xFAD4,5xFAD,23372,6671,6655,15.5
5xFAD5,5xFAD,18118,5208,6674,16.2
Trem2_1,Trem2,28045,6021,6646,13.2
Trem2_2,Trem2,14054,3585,4601,13.2
Trem2_3,Trem2,22019,5610,6031,12.2
Trem2_4,Trem2,25617,6022,6994,14.4


## 7. Write the file

- Save to a temporary file first.
- If `trem2_5xfad.h5ad` does not exist, rename the temporary file to it.
- If it exists, compare cell order, genes, coordinates, all `obs` values, and expression data—including exact values for 5,000 random cells.
- If all checks pass, replace the old file in one step so readers never see a partially written file.
- If anything differs, keep the old file, save the new one as `trem2_5xfad.new.h5ad`, and list the differences.

In [8]:
def compare_h5ad(new_path, ref_path, n_check=5000, seed=0):
    """Every difference found between two h5ad files; an empty list means they agree on every check."""
    new, ref = ad.read_h5ad(new_path), ad.read_h5ad(ref_path)
    diffs = []
    if not new.obs_names.equals(ref.obs_names):
        diffs.append(f"cells differ in identity or order ({new.n_obs:,} vs {ref.n_obs:,})")
    if not new.var_names.equals(ref.var_names):
        diffs.append(f"genes differ in identity or order ({new.n_vars:,} vs {ref.n_vars:,})")
    if diffs:                                   # element-wise checks need matching rows and columns
        return diffs
    Xn, Xr = new.X.tocsr(), ref.X.tocsr()
    if Xn.dtype != Xr.dtype:
        diffs.append(f"X dtype: {Xn.dtype} vs {Xr.dtype}")
    if Xn.nnz != Xr.nnz:
        diffs.append(f"X non-zeros: {Xn.nnz:,} vs {Xr.nnz:,}")
    sn, sr = Xn.data.sum(dtype=np.float64), Xr.data.sum(dtype=np.float64)
    if sn != sr:
        diffs.append(f"X total: {sn!r} vs {sr!r}")
    rows = np.sort(np.random.default_rng(seed).choice(new.n_obs, min(n_check, new.n_obs), replace=False))
    n_diff = (Xn[rows] != Xr[rows]).nnz
    if n_diff:
        diffs.append(f"X differs in {n_diff:,} entries of {len(rows):,} random cells")
    sn_, sr_ = np.asarray(new.obsm["spatial"]), np.asarray(ref.obsm["spatial"])
    if sn_.dtype != sr_.dtype or sn_.shape != sr_.shape or not np.array_equal(sn_, sr_):
        diffs.append("spatial coordinates differ")
    if list(new.obs.columns) != list(ref.obs.columns):
        diffs.append(f"obs columns: {list(new.obs.columns)} vs {list(ref.obs.columns)}")
    for c in new.obs.columns.intersection(ref.obs.columns):
        x, y = new.obs[c], ref.obs[c]
        if x.dtype != y.dtype:
            diffs.append(f"obs['{c}'] dtype: {x.dtype} vs {y.dtype}")
        if x.dtype.kind in "biuf" and y.dtype.kind in "biuf":
            same = np.array_equal(x.to_numpy(), y.to_numpy(), equal_nan=True)
        else:
            same = np.array_equal(x.astype(str).to_numpy(), y.astype(str).to_numpy())
        if not same:
            diffs.append(f"obs['{c}'] values differ")
    return diffs

In [9]:
adata.write_h5ad(TMP, compression="gzip")
if not os.path.exists(OUT):
    os.replace(TMP, OUT)
    print("wrote", short(OUT))
else:
    diffs = compare_h5ad(TMP, OUT)
    if not diffs:
        os.replace(TMP, OUT)
        print(f"identical to the existing {os.path.basename(OUT)} on every check; replaced it")
    else:
        os.replace(TMP, NEW)
        print(f"differs from the existing {os.path.basename(OUT)}; kept the existing file and wrote", short(NEW))
        for d in diffs:
            print("  -", d)

identical to the existing trem2_5xfad.h5ad on every check; replaced it
